 Capillary Bridge Force Analysis Workflow

# 1. Setup and Imports

In [ ]:
# If running first time, uncomment to install dependencies
# !pip install -r requirements.txt

import os
import sys
from pathlib import Path

# Add project root to path if needed
project_root = Path(".").resolve()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

# Core modules
from Analysis_Files.Substrate_tests.define_substrate_lines import pick_and_fit_lines
from Analysis_Files.Final_RunFile_Working import analyze_frame_sequence
from Analysis_Files.plot_forces_multiprocessing import run_full_analysis
import Analysis_Files.config


# 2. Configuration

In [ ]:
# Paths (modify to match your directory structure)
config.output_dir = project_root / "results"
config.top_lines_file = config.output_dir / "substrate_lines_top.npy"
config.bottom_lines_file = config.output_dir / "substrate_lines_bottom.npy"

# Classifier paths
mask_edge_clf_path = project_root / "classifiers" / "mask_edge_classifier2.pkl"
side_clf_path      = project_root / "classifiers" / "contour_side_classifier2.pkl"

# Create output directory
os.makedirs(config.output_dir, exist_ok=True)


# 3. Substrate-Line Definition

In [ ]:
# This will open an interactive tool to click substrate lines on sample frames
# and interpolate for all frames automatically.
pick_and_fit_lines(
    image_dir = project_root / "data" / "images",
    first_frame = config.first_frame,
    last_frame  = config.last_frame,
    spacing     = config.first_frame_spacing,
    top_save    = config.top_lines_file,
    bottom_save = config.bottom_lines_file
)


# 4. Frame-by-Frame Analysis

In [ ]:
results = analyze_frame_sequence(
    mask_dir            = config.mask_dir,
    top_lines_file      = config.top_lines_file,
    bottom_lines_file   = config.bottom_lines_file,
    mask_edge_clf_path  = mask_edge_clf_path,
    side_clf_path       = side_clf_path,
    output_prefix       = config.output_dir / "frame_results"
)

# The `results` DataFrame contains per-frame metrics including separation, angles, curvature, and force.
# Display first few rows:
results.head()


# 5. Multiprocessing Orchestration & Final Plot

In [ ]:
excel_path, fig = run_full_analysis(
    mask_dir            = config.mask_dir,
    top_lines_file      = config.top_lines_file,
    bottom_lines_file   = config.bottom_lines_file,
    mask_edge_clf_path  = mask_edge_clf_path,
    side_clf_path       = side_clf_path,
    output_excel        = config.output_dir / "capillary_forces.xlsx",
    output_plot         = config.output_dir / "force_vs_separation.png",
    n_workers           = 4
)

print(f"Results saved to Excel: {excel_path}")
print(f"Plot saved to: {config.output_dir / 'force_vs_separation.png'}")


# 6. Display Final Plot

In [ ]:
from IPython.display import Image
Image(filename=str(config.output_dir / "force_vs_separation.png"))


## Next Steps